### Currently (June 2026), I am finding that critic networks in the rl algorithms may be having difficulty generalizing across patient states, and this is making it difficult for the actors to identify actions that lead to higher Q values. 

I suspect this may be due to the high dimensionality of the transformer latents I am using as the input states to the RL training loop - these latents come from both dynamic deephit (dim 512), and the behavioural clone (dim 256).

This notebook uses SVD to determine the number of components to explain most of the variance, and then keeps only the minimum number of essesntial dimensions 

In [5]:
import os
import h5py

In [9]:
dynamic_deephit_predictions_path = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/"
bc_transformer_final_model_dir = "/home/jake/Documents/AI_Clinician/models/bc_transformer/choc/"
cohort = "sepsis"
actions = "intake-vis"
split = 0

In [ ]:
dynamic_deephit_predictions_path = os.path.join(dynamic_deephit_predictions_path, 
    "five_splits", f"split_{0}", f"{cohort}", f"{actions}", "final_model")

# find the folder inside the final_model directory
final_model_dir = os.listdir(dynamic_deephit_predictions_path)[0]
print(final_model_dir)

choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260326_040032_999109_job2024209.pbs-7


In [ ]:
dynamic_deephit_train_predictions_path = os.path.join(final_model_dir,
    "train_set_predictions.h5")

In [ ]:
import torch

latents_tensor = torch.tensor(self.dynamic_deephit_predictions["model_outputs"], dtype=torch.float32)

latents_tensor = torch.tensor(self.bc_latents["model_outputs"], dtype=torch.float32)



# Center
latents_tensor -= latents_tensor.mean(dim=0)

U, S, Vh = torch.linalg.svd(latents_tensor, full_matrices=False)

explained = (S**2).cumsum(0) / (S**2).sum()

for k in [10, 20, 30, 50, 100, 200]:
    print(f"Top {k:3d} components: {explained[k-1]*100:.1f}%")